# Preparation of expression-analysis input files

This notebook prepares the count matrix and gene-annotation table used by `01_differential_expression.R`.

Main outputs used in the next step:

- `ldec_dge_counts.tsv`
- `ldec_gff_annotated.tsv`

`Design_table.tsv` is used by `01_differential_expression.R` as a separate sample-metadata input and is not regenerated in this notebook.


In [ ]:
import os
import pandas as pd
import glob
import time, re
from itertools import repeat
import numpy as np

In [ ]:
import matplotlib.pyplot as plt # библиотека для построения графиков
import seaborn as sns # библиотека для построения графиков

In [ ]:
import scipy
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import fdrcorrection as fdr

## 1. Combine STAR count tables


In [ ]:
tabs = glob.glob('./Ldec2.0_counts-star/*.tab')

In [ ]:
dge = pd.concat ( [
    pd.read_table( x , header = None, index_col = 0 ).iloc[:,0] for x in tabs ] , axis = 1)

In [ ]:
dge.columns = [ x.split('\\')[-1].split('_')[0] for x in tabs ]

In [ ]:
dge

In [ ]:
dge_sel = dge.drop(index = 'MissingGeneID')#.drop(index='')

In [ ]:
dge_sel.index.value_counts()

In [ ]:
pd.concat ( [ dge.sum(), dge_sel.sum(), dge_sel.iloc[4:, :].sum(), dge.loc['MissingGeneID'] ] , axis = 1)

## 2. Read Ldec_2.0 genome annotation


In [ ]:
from gtfparse import read_gtf

In [ ]:
ldec_gtf = read_gtf('./Ldec_2.0/GCF_000500325.1_Ldec_2.0_genomic.gtf')

In [ ]:
ldec_gtf

In [ ]:
ldec_cds = ldec_gtf[ldec_gtf.feature.eq('CDS')]

In [ ]:
pd.Series( ldec_gtf.gene_id.unique() ).to_csv('ldec_unique_gene_ids.txt', index = False)

In [ ]:
ldec_cds.gene_id#[ldec_cds.gene_id.str.contains('LOC')]

In [ ]:
pd.Series( dge.index[4:].isin( ldec_gtf.gene_id ) ) .value_counts()

In [ ]:
dge.index[4:] [ ~dge.index[4:].isin( ldec_gtf.gene_id ) ]

In [ ]:
dge.index[4:] [ ~dge.index[4:].isin( ldec_cds.gene_id ) ]

In [ ]:
with open('./GCF_000500325.1_Ldec_2.0_protein.gpff') as f:
    recs = f.read()
prots = re.findall('^(LOCUS.+?)^//', recs, re.MULTILINE + re.DOTALL)

In [ ]:
len(prots)

In [ ]:
#sel_prots = [x for x in prots if '/gene="LOC' in x]

In [ ]:
#print(sel_prots[100])

In [ ]:
ldec_gff = read_gtf('./Ldec_2.0/GCF_000500325.1_Ldec_2.0_genomic.gff')

In [ ]:
import gc
gc.collect()

## 3. Parse GFF3 annotation and prepare gene products


In [ ]:
import gff3_parser

In [ ]:
ldec_gff = gff3_parser.parse_gff3('./Ldec_2.0/GCF_000500325.1_Ldec_2.0_genomic.gff', verbose = False,
                                  parse_attributes = True )

In [ ]:
ldec_gff.columns

In [ ]:
ldec_gff.Type.value_counts()

In [ ]:
ldec_gff[ldec_gff.Type.eq('CDS')].to_csv('ldec_gff_cds.tsv', sep = '\t')

In [ ]:
ldec_gff[ldec_gff.Type.eq('CDS')].Dbxref[ldec_gff[ldec_gff.Type.eq('CDS')].Dbxref.str.contains('pfam')]

In [ ]:
ldec_gff[ldec_gff.Type.eq('CDS')][['gene', 'product']]

In [ ]:
ldec_gtf = ldec_gtf.rename(columns = {'gene_name':'gene'})

In [ ]:
gene_prod = ldec_gff[['gene', 'product']].dropna().drop_duplicates()#.fillna('').drop_duplicates()

In [ ]:
gset1 = gene_prod[~gene_prod['product'].str.contains('%')].groupby('gene').agg(lambda x: '; '.join(x))
gset2 = gene_prod[gene_prod['product'].str.contains('%')].groupby('gene').agg(lambda x: '; '.join(x))


In [ ]:
gset = pd.concat( [gset1, gset2.loc[list(set(gset2.index.values) - set(gset1.index.values))] ]).reset_index()

In [ ]:
ldec_gtf.merge( gset, how = 'left')

## 4. Write files used by differential expression analysis


In [ ]:
ldec_gtf.merge( gset, how = 'left').to_csv('ldec_gff_annotated.tsv', sep = '\t')

In [ ]:
dge_sel.iloc[4:,:].to_csv('ldec_dge_counts.tsv', sep ='\t')